# Kuzu Graph Database Integration

This notebook demonstrates how to work with topologic_fast graphs and Kuzu, an embedded graph database.

**Note:** Native Kuzu integration is not yet implemented in topologic_fast. This notebook shows:
1. How to create topological graphs using topologic_fast
2. How to extract graph data for export to Kuzu
3. How to create a Kuzu database and import the graph

## What is Kuzu?

Kuzu is an embeddable property graph database management system built for query speed and scalability. Key features:
- **Embedded**: No server required, runs in-process
- **Fast**: Optimized for graph analytics
- **Cypher Support**: Uses openCypher query language
- **Python Native**: First-class Python support

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import json
import os
import shutil

# Check if kuzu is available
try:
    import kuzu
    KUZU_AVAILABLE = True
    print(f"Kuzu version: {kuzu.__version__}")
except ImportError:
    KUZU_AVAILABLE = False
    print("Kuzu not installed. Install with: pip install kuzu")
    print("This notebook will still demonstrate the graph structure for export.")

## 1. Create a Sample Graph

Let's create a CellComplex and generate its dual graph.

In [ ]:
# Create a 3x3x3 grid of cells
cells = []
cell_size = 2.0

for i in range(3):
    for j in range(3):
        for k in range(3):
            cell = tf.Cell.Box(
                i * cell_size, j * cell_size, k * cell_size,
                cell_size, cell_size, cell_size
            )
            cells.append(cell)

# Create CellComplex
cell_complex = tf.CellComplex.ByCells(cells)

# Create dual graph
sample_graph = tf.Graph.ByTopology(cell_complex)

print(f"CellComplex Statistics:")
print(f"  Cells: {cell_complex.NumCells()}")
print(f"  Volume: {cell_complex.Volume():.1f}")

print(f"\nGraph Statistics:")
print(f"  Vertices: {sample_graph.Order()}")
print(f"  Edges: {sample_graph.Size()}")
print(f"  Density: {sample_graph.Density():.4f}")
print(f"  Diameter: {sample_graph.Diameter()}")

## 2. Create Vertex Metadata

Assign labels and properties to each vertex.

In [ ]:
# NOTE: tf.Dictionary is not yet implemented in topologic_fast
# We'll track metadata separately

graph_vertices = sample_graph.Vertices()

# Create metadata for each vertex
vertex_metadata = []
for i, v in enumerate(graph_vertices):
    coords = v.Coordinates()
    
    # Determine grid position
    grid_x = int(coords[0] / cell_size)
    grid_y = int(coords[1] / cell_size)
    grid_z = int(coords[2] / cell_size)
    
    vertex_metadata.append({
        "id": i,
        "label": f"Cell_{i+1}",
        "x": coords[0],
        "y": coords[1],
        "z": coords[2],
        "grid_x": grid_x,
        "grid_y": grid_y,
        "grid_z": grid_z,
        "level": grid_z  # Floor level
    })

print(f"Created metadata for {len(vertex_metadata)} vertices")
print("\nFirst 5 vertices:")
for v in vertex_metadata[:5]:
    print(f"  {v['label']}: grid=({v['grid_x']},{v['grid_y']},{v['grid_z']})")

## 3. Extract Edge Data

In [ ]:
def find_vertex_index(target_coords, vertices, tolerance=0.01):
    """Find vertex index by coordinates."""
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        if (abs(coords[0] - target_coords[0]) < tolerance and
            abs(coords[1] - target_coords[1]) < tolerance and
            abs(coords[2] - target_coords[2]) < tolerance):
            return i
    return -1

# Extract edges
graph_edges = sample_graph.Edges()
edge_data = []

for edge in graph_edges:
    edge_verts = edge.Vertices()
    if len(edge_verts) == 2:
        coords1 = edge_verts[0].Coordinates()
        coords2 = edge_verts[1].Coordinates()
        
        idx1 = find_vertex_index(coords1, graph_vertices)
        idx2 = find_vertex_index(coords2, graph_vertices)
        
        if idx1 >= 0 and idx2 >= 0:
            # Determine relationship type
            z1 = vertex_metadata[idx1]['grid_z']
            z2 = vertex_metadata[idx2]['grid_z']
            
            if z1 != z2:
                rel_type = "VERTICAL"
            else:
                rel_type = "HORIZONTAL"
            
            edge_data.append({
                "from_id": idx1,
                "to_id": idx2,
                "type": rel_type,
                "from_label": vertex_metadata[idx1]['label'],
                "to_label": vertex_metadata[idx2]['label']
            })

print(f"Extracted {len(edge_data)} edges")
print("\nSample edges:")
for e in edge_data[:5]:
    print(f"  ({e['from_label']})-[:{e['type']}]->({e['to_label']})")

## 4. Visualize the Graph

In [ ]:
def visualize_graph(graph, metadata, title="Graph Visualization"):
    """Create 3D visualization of the graph."""
    fig = go.Figure()
    
    # Draw edges
    edges = graph.Edges()
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            # Color by edge type
            if abs(p1[2] - p2[2]) > 0.1:
                color = 'red'  # Vertical
            else:
                color = 'blue'  # Horizontal
            
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color=color, width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    vertices = graph.Vertices()
    x = [m['x'] for m in metadata]
    y = [m['y'] for m in metadata]
    z = [m['z'] for m in metadata]
    colors = [m['level'] for m in metadata]  # Color by level
    labels = [m['label'] for m in metadata]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=10,
            color=colors,
            colorscale='Viridis',
            colorbar=dict(title='Level'),
            line=dict(color='black', width=1)
        ),
        text=labels,
        hovertext=[f"{m['label']}\nLevel: {m['level']}" for m in metadata],
        hoverinfo='text',
        name='Vertices'
    ))
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        width=800,
        height=600
    )
    
    return fig

fig = visualize_graph(sample_graph, vertex_metadata, "3x3x3 Cell Grid - Dual Graph")
fig.show()

## 5. Create Kuzu Database

Now let's create a Kuzu database and import the graph.

In [ ]:
if KUZU_AVAILABLE:
    # Database path
    db_path = "./topologic_graph.kuzu"
    
    # Remove existing database if present
    if os.path.exists(db_path):
        shutil.rmtree(db_path)
    
    # Create database
    db = kuzu.Database(db_path)
    conn = kuzu.Connection(db)
    
    print(f"Created Kuzu database at: {db_path}")
else:
    print("Kuzu not available. Showing schema definition only.")

## 6. Define Schema and Import Data

In [ ]:
# Schema definition
schema_ddl = """
-- Create node table for cells
CREATE NODE TABLE Cell (
    id INT64 PRIMARY KEY,
    label STRING,
    x DOUBLE,
    y DOUBLE,
    z DOUBLE,
    grid_x INT64,
    grid_y INT64,
    grid_z INT64,
    level INT64
);

-- Create edge table for horizontal adjacency
CREATE REL TABLE HORIZONTAL (
    FROM Cell TO Cell
);

-- Create edge table for vertical adjacency
CREATE REL TABLE VERTICAL (
    FROM Cell TO Cell
);
"""

print("Schema DDL:")
print(schema_ddl)

if KUZU_AVAILABLE:
    # Execute schema creation
    conn.execute("CREATE NODE TABLE Cell (id INT64 PRIMARY KEY, label STRING, x DOUBLE, y DOUBLE, z DOUBLE, grid_x INT64, grid_y INT64, grid_z INT64, level INT64)")
    conn.execute("CREATE REL TABLE HORIZONTAL (FROM Cell TO Cell)")
    conn.execute("CREATE REL TABLE VERTICAL (FROM Cell TO Cell)")
    print("\nSchema created successfully!")

In [ ]:
if KUZU_AVAILABLE:
    # Insert vertices
    print("Inserting vertices...")
    for v in vertex_metadata:
        conn.execute(
            "CREATE (c:Cell {id: $id, label: $label, x: $x, y: $y, z: $z, "
            "grid_x: $grid_x, grid_y: $grid_y, grid_z: $grid_z, level: $level})",
            {"id": v["id"], "label": v["label"], 
             "x": v["x"], "y": v["y"], "z": v["z"],
             "grid_x": v["grid_x"], "grid_y": v["grid_y"], 
             "grid_z": v["grid_z"], "level": v["level"]}
        )
    print(f"  Inserted {len(vertex_metadata)} vertices")
    
    # Insert edges
    print("Inserting edges...")
    h_count = 0
    v_count = 0
    
    for e in edge_data:
        if e["type"] == "HORIZONTAL":
            conn.execute(
                "MATCH (a:Cell {id: $from_id}), (b:Cell {id: $to_id}) "
                "CREATE (a)-[:HORIZONTAL]->(b)",
                {"from_id": e["from_id"], "to_id": e["to_id"]}
            )
            h_count += 1
        else:
            conn.execute(
                "MATCH (a:Cell {id: $from_id}), (b:Cell {id: $to_id}) "
                "CREATE (a)-[:VERTICAL]->(b)",
                {"from_id": e["from_id"], "to_id": e["to_id"]}
            )
            v_count += 1
    
    print(f"  Inserted {h_count} HORIZONTAL edges")
    print(f"  Inserted {v_count} VERTICAL edges")
    print("\nData import complete!")
else:
    print("Kuzu not available. Sample insert statements:")
    print("\n-- Insert vertices")
    for v in vertex_metadata[:3]:
        print(f"CREATE (c:Cell {{id: {v['id']}, label: '{v['label']}', level: {v['level']}}});")
    print("...")

## 7. Query the Graph

In [ ]:
if KUZU_AVAILABLE:
    print("Query 1: Count all cells")
    result = conn.execute("MATCH (c:Cell) RETURN count(c) AS total")
    while result.has_next():
        print(f"  Total cells: {result.get_next()[0]}")
    
    print("\nQuery 2: Cells on level 0")
    result = conn.execute("MATCH (c:Cell) WHERE c.level = 0 RETURN c.label ORDER BY c.id")
    cells_level_0 = []
    while result.has_next():
        cells_level_0.append(result.get_next()[0])
    print(f"  {', '.join(cells_level_0)}")
    
    print("\nQuery 3: Cells with most horizontal connections")
    result = conn.execute(
        "MATCH (c:Cell)-[:HORIZONTAL]-(neighbor) "
        "RETURN c.label, count(neighbor) AS connections "
        "ORDER BY connections DESC LIMIT 5"
    )
    while result.has_next():
        row = result.get_next()
        print(f"  {row[0]}: {row[1]} connections")
    
    print("\nQuery 4: Shortest path between Cell_1 and Cell_27")
    result = conn.execute(
        "MATCH (a:Cell {label: 'Cell_1'}), (b:Cell {label: 'Cell_27'}), "
        "path = (a)-[*BFS]-(b) "
        "RETURN length(path) AS path_length"
    )
    while result.has_next():
        print(f"  Path length: {result.get_next()[0]}")
else:
    print("Example Cypher queries for Kuzu:")
    queries = [
        "-- Count all cells\nMATCH (c:Cell) RETURN count(c)",
        "-- Find cells on level 0\nMATCH (c:Cell) WHERE c.level = 0 RETURN c.label",
        "-- Find horizontally adjacent cells\nMATCH (a:Cell)-[:HORIZONTAL]-(b:Cell) RETURN a.label, b.label",
        "-- Find vertically stacked cells\nMATCH (a:Cell)-[:VERTICAL]-(b:Cell) RETURN a.label, b.label",
        "-- Shortest path\nMATCH (a:Cell {label: 'Cell_1'}), (b:Cell {label: 'Cell_27'}),\n      path = (a)-[*BFS]-(b)\nRETURN path"
    ]
    for q in queries:
        print(f"\n{q}")

## 8. Create and Export a Subgraph

In [ ]:
# Create an induced subgraph using topologic_fast
# Get vertices for level 0 and level 1 only
subgraph_vertices = []
for i, m in enumerate(vertex_metadata):
    if m['level'] <= 1:  # Only levels 0 and 1
        subgraph_vertices.append(graph_vertices[i])

print(f"Selected {len(subgraph_vertices)} vertices for subgraph")

# Create edges connecting these vertices
subgraph_edges = []
subgraph_vertices_set = set(range(len([m for m in vertex_metadata if m['level'] <= 1])))

for e in edge_data:
    from_level = vertex_metadata[e['from_id']]['level']
    to_level = vertex_metadata[e['to_id']]['level']
    if from_level <= 1 and to_level <= 1:
        subgraph_edges.append(e)

print(f"Subgraph has {len(subgraph_edges)} edges")

In [ ]:
# Visualize the subgraph
subgraph_metadata = [m for m in vertex_metadata if m['level'] <= 1]

fig = go.Figure()

# Draw edges
for e in subgraph_edges:
    p1 = (vertex_metadata[e['from_id']]['x'], 
          vertex_metadata[e['from_id']]['y'],
          vertex_metadata[e['from_id']]['z'])
    p2 = (vertex_metadata[e['to_id']]['x'],
          vertex_metadata[e['to_id']]['y'],
          vertex_metadata[e['to_id']]['z'])
    
    color = 'red' if e['type'] == 'VERTICAL' else 'blue'
    fig.add_trace(go.Scatter3d(
        x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
        mode='lines',
        line=dict(color=color, width=3),
        showlegend=False
    ))

# Draw vertices
x = [m['x'] for m in subgraph_metadata]
y = [m['y'] for m in subgraph_metadata]
z = [m['z'] for m in subgraph_metadata]

fig.add_trace(go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers+text',
    marker=dict(size=12, color=[m['level'] for m in subgraph_metadata], colorscale='Viridis'),
    text=[m['label'] for m in subgraph_metadata],
    textposition='top center',
    name='Cells'
))

fig.update_layout(
    title='Subgraph: Levels 0-1 Only',
    scene=dict(aspectmode='data'),
    width=800,
    height=600
)

fig.show()

## 9. Export Graph for Kuzu CSV Import

In [ ]:
import csv

# Export vertices to CSV
with open('./cells.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'label', 'x', 'y', 'z', 'grid_x', 'grid_y', 'grid_z', 'level'])
    writer.writeheader()
    writer.writerows(vertex_metadata)

print("Exported cells.csv")

# Export horizontal edges to CSV
h_edges = [e for e in edge_data if e['type'] == 'HORIZONTAL']
with open('./horizontal_edges.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['from', 'to'])
    for e in h_edges:
        writer.writerow([e['from_id'], e['to_id']])

print(f"Exported horizontal_edges.csv ({len(h_edges)} edges)")

# Export vertical edges to CSV
v_edges = [e for e in edge_data if e['type'] == 'VERTICAL']
with open('./vertical_edges.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['from', 'to'])
    for e in v_edges:
        writer.writerow([e['from_id'], e['to_id']])

print(f"Exported vertical_edges.csv ({len(v_edges)} edges)")

# Show Kuzu COPY commands
print("\n-- Kuzu COPY commands for bulk import:")
print("COPY Cell FROM 'cells.csv';")
print("COPY HORIZONTAL FROM 'horizontal_edges.csv';")
print("COPY VERTICAL FROM 'vertical_edges.csv';")

## 10. Cleanup

In [ ]:
if KUZU_AVAILABLE:
    # Close connection
    del conn
    del db
    print("Closed Kuzu database connection")
    
    # Optionally remove the database
    # shutil.rmtree(db_path)
    # print(f"Removed database at {db_path}")

print("\nGenerated files:")
print("  - cells.csv")
print("  - horizontal_edges.csv")
print("  - vertical_edges.csv")
if KUZU_AVAILABLE:
    print(f"  - {db_path}/ (Kuzu database)")

## Summary

This notebook demonstrated:

1. **Graph Creation** - Building a 3x3x3 grid CellComplex and its dual graph
2. **Metadata Assignment** - Tracking vertex properties (position, level)
3. **Edge Classification** - Distinguishing HORIZONTAL vs VERTICAL relationships
4. **Kuzu Integration** - Creating schema, importing data, and querying
5. **CSV Export** - Generating files for bulk import

### Features Not Yet Implemented in topologic_fast:

- `tf.Kuzu` - Native Kuzu integration class
- `tf.Dictionary` - Attaching metadata to topologies
- `tf.Graph.UpsertGraph()` - Upserting graphs to database
- `tf.Graph.InducedSubgraph()` - Creating subgraphs (available but shown manually)
- `tf.Graph.KHopsSubgraph()` - K-hops neighborhood extraction

### Advantages of Kuzu:

- **Embedded**: No server setup required
- **Fast**: Optimized for graph analytics
- **Cypher**: Familiar query language
- **Python Native**: Easy integration with data science workflows